# **EduSense: Stage A — Feature Extraction (AffectNet-Pretrained)**

**King Khalid University — Graduation Project 2025**

---

## Why We Changed the Feature Extractor

| Extractor | Pretrained On | Emotion-Specific? | Embedding Dim |
|-----------|--------------|-------------------|---------------|
| `google/vit-base-patch16-224-in21k` ❌ | ImageNet (general) | ❌ No | 768 |
| `motheecreator/vit-Facial-Expression-Recognition` ✅ | AffectNet (facial emotions) | ✅ Yes | 768 |

Using a ViT pretrained specifically on **AffectNet** (460K facial expression images) gives the model emotion-aware features from the start — no need to learn faces from scratch.

---

## Pipeline

```
DAISEE Videos
     ↓
YOLO (face detection & crop)
     ↓
AffectNet-ViT (frozen, emotion-aware features)
     ↓
Save .npy embeddings  →  Used in Stage B (LSTM → KAN → CORAL)
```

**⚠️ Run this notebook ONCE. It saves embeddings to disk.**

## 📦 **1. Install Dependencies**

In [ ]:
!pip install -q ultralytics transformers opencv-python-headless tqdm kagglehub Pillow
print('✅ Dependencies installed')

## 📚 **2. Imports**

In [ ]:
import os
import json
import glob
import shutil
import numpy as np
import pandas as pd
import cv2
import torch
from pathlib import Path
from PIL import Image
from tqdm.notebook import tqdm
from transformers import ViTModel, ViTImageProcessor
from ultralytics import YOLO
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 📥 **3. Download DAISEE Dataset**

In [ ]:
import kagglehub

path = kagglehub.dataset_download('olgaparfenova/daisee')
print(f'Downloaded to: {path}')

DATA_ROOT = '/content/DAiSEE'
shutil.copytree(path, DATA_ROOT, dirs_exist_ok=True)

print('\nDAiSEE structure:')
for item in os.listdir(DATA_ROOT):
    print(f'  {item}')

## 🏷️ **4. Load Labels from CSV**

In [ ]:
def load_daisee_labels(data_root):
    """
    Load DAISEE labels from CSV files.

    DAISEE CSV format:
      ClipID            | Engagement | Boredom | Confusion | Frustration
      110001010101.avi  | 2          | 0       | 1         | 0

    Returns dict: {clip_id_with_ext: {engagement, boredom, confusion, frustration, split}}
    e.g. key = '110001010101.avi'
    """
    label_dict = {}

    for split in ['Train', 'Val', 'Test']:
        candidates = [
            os.path.join(data_root, 'DAiSEE', 'Labels', f'{split}Labels.csv'),
            os.path.join(data_root, 'Labels', f'{split}Labels.csv'),
        ]
        csv_path = next((p for p in candidates if os.path.exists(p)), None)

        if csv_path:
            df = pd.read_csv(csv_path)
            print(f'{split}: {len(df)} clips | columns: {df.columns.tolist()}')

            for _, row in df.iterrows():
                clip_id = str(row['ClipID']).strip()  # e.g. '110001010101.avi'
                label_dict[clip_id] = {
                    'engagement':  int(row['Engagement']),
                    'boredom':     int(row['Boredom']),
                    'confusion':   int(row['Confusion']),
                    'frustration': int(row['Frustration']),
                    'split':       split.lower()
                }
        else:
            print(f'WARNING: CSV not found for split: {split}')

    print(f'\n✅ Total labels loaded: {len(label_dict)}')
    return label_dict


label_dict = load_daisee_labels(DATA_ROOT)

sample_key = list(label_dict.keys())[0]
print(f'\nSample entry:')
print(f'  ClipID: {sample_key}')  # Should look like '110001010101.avi'
print(f'  Labels: {label_dict[sample_key]}')


## 🔍 **5. Find All Video Paths + Match Labels**

In [ ]:
def find_videos_with_labels(data_root, label_dict):
    """
    Build video paths using exact DAISEE folder structure:
      DataSet/{split}/{subject_id}/{clip_folder}/{clip_id}.avi

    Where:
      subject_id   = clip_id[:6]           e.g. '110001'
      clip_folder  = clip_id without .avi  e.g. '110001010101'
      clip_id      = full name             e.g. '110001010101.avi'
    """
    split_map = {'train': 'Train', 'val': 'Val', 'test': 'Test'}

    # Try both possible dataset roots
    dataset_root_candidates = [
        os.path.join(data_root, 'DAiSEE', 'DataSet'),
        os.path.join(data_root, 'DataSet'),
    ]
    dataset_root = next((p for p in dataset_root_candidates if os.path.exists(p)), None)
    if dataset_root is None:
        print('ERROR: Cannot find DataSet folder!')
        return []
    print(f'Dataset root: {dataset_root}')

    matched = []
    missing = 0

    for clip_id, labels in label_dict.items():
        split_folder = split_map.get(labels['split'], labels['split'].capitalize())
        subject_id  = clip_id[:6]                    # e.g. '110001'
        clip_folder = clip_id.replace('.avi', '')    # e.g. '110001010101'

        video_path = os.path.join(
            dataset_root,
            split_folder,
            subject_id,
            clip_folder,
            clip_id
        )

        if os.path.exists(video_path):
            matched.append({
                'video_path':  video_path,
                'clip_name':   clip_folder,  # use stem (no .avi) as save filename
                'engagement':  labels['engagement'],
                'boredom':     labels['boredom'],
                'confusion':   labels['confusion'],
                'frustration': labels['frustration'],
                'split':       labels['split']
            })
        else:
            missing += 1

    splits = {}
    for item in matched:
        s = item['split']
        splits[s] = splits.get(s, 0) + 1

    print(f'Matched: {len(matched)} | Missing on disk: {missing}')
    print(f'Split distribution: {splits}')
    return matched


all_videos = find_videos_with_labels(DATA_ROOT, label_dict)
print(f'\n✅ Total matched videos: {len(all_videos)}')

# Show one example path
if all_videos:
    print(f'\nExample path: {all_videos[0]["video_path"]}')
    print(f'Labels: {all_videos[0]}')


## 🎯 **6. YOLO Face Detector**

In [ ]:
class YOLOFaceDetector:
    """
    YOLOv8-based face detector.
    Detects and crops faces from video frames.
    """
    
    def __init__(self, model_path='yolov8n.pt', conf_threshold=0.4, device='cuda'):
        # Use face-specific model if available, otherwise general
        self.model = YOLO(model_path)
        self.conf_threshold = conf_threshold
        self.device = device
        print(f'✅ YOLO loaded: {model_path}')
    
    def detect_and_crop(self, frame, padding=0.2, target_size=(224, 224)):
        """
        Detect face, crop with padding, resize to target_size.
        
        Returns:
            face_crop: PIL Image (224, 224) or None if no face
        """
        results = self.model(frame, conf=self.conf_threshold, verbose=False)
        
        best_box = None
        best_area = 0
        
        for result in results:
            for box in result.boxes:
                # Check if it's a person/face class
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                area = (x2 - x1) * (y2 - y1)
                if area > best_area:
                    best_area = area
                    best_box = (x1, y1, x2, y2)
        
        if best_box is None:
            return None
        
        x1, y1, x2, y2 = best_box
        h, w = frame.shape[:2]
        
        # Add padding
        pad_x = (x2 - x1) * padding
        pad_y = (y2 - y1) * padding
        x1 = max(0, int(x1 - pad_x))
        y1 = max(0, int(y1 - pad_y))
        x2 = min(w, int(x2 + pad_x))
        y2 = min(h, int(y2 + pad_y))
        
        # Crop and convert BGR -> RGB
        crop = frame[y1:y2, x1:x2]
        crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
        pil_img = Image.fromarray(crop_rgb).resize(target_size)
        
        return pil_img


yolo_detector = YOLOFaceDetector(device=str(device))
print('✅ YOLO Face Detector ready')

## 🧠 **7. AffectNet-Pretrained ViT Feature Extractor**

> **Why this model?**  
> `motheecreator/vit-Facial-Expression-Recognition` is a ViT fine-tuned on **AffectNet** — 460,000 real-world facial expression images labeled with 8 emotions (neutral, happy, sad, surprise, fear, disgust, anger, contempt).  
> It understands faces and emotions from the start. The generic ImageNet-ViT does not.

In [ ]:
class AffectNetViTExtractor:
    """
    AffectNet-pretrained Vision Transformer for facial emotion feature extraction.
    
    Model: motheecreator/vit-Facial-Expression-Recognition
    Pretrained on: AffectNet (460K facial expressions, 8 emotions)
    Output: 768-dim CLS token embedding (emotion-aware)
    
    FROZEN — no gradients, used only for feature extraction.
    """
    
    MODEL_NAME = 'motheecreator/vit-Facial-Expression-Recognition'
    
    def __init__(self, device='cuda'):
        self.device = torch.device(device)
        
        print(f'Loading AffectNet-ViT: {self.MODEL_NAME}')
        print('This model was pretrained on AffectNet facial expressions.')
        
        self.processor = ViTImageProcessor.from_pretrained(self.MODEL_NAME)
        self.model = ViTModel.from_pretrained(self.MODEL_NAME).to(self.device)
        
        # FREEZE — we only want features, not to fine-tune
        self.model.eval()
        for param in self.model.parameters():
            param.requires_grad = False
        
        self.embedding_dim = self.model.config.hidden_size  # 768
        print(f'✅ AffectNet-ViT loaded (FROZEN) | Embedding dim: {self.embedding_dim}')
    
    @torch.no_grad()
    def extract(self, pil_image):
        """
        Extract 768-dim CLS token from face image.
        
        Args:
            pil_image: PIL Image (RGB, any size — processor handles resize)
        
        Returns:
            embedding: numpy array (768,)
        """
        inputs = self.processor(images=pil_image, return_tensors='pt')
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        
        outputs = self.model(**inputs)
        
        # CLS token = first token, summarizes whole image
        cls_embedding = outputs.last_hidden_state[:, 0, :]  # (1, 768)
        
        return cls_embedding.squeeze(0).cpu().numpy()  # (768,)


affectnet_extractor = AffectNetViTExtractor(device=str(device))
print(f'\nEmbedding dimension: {affectnet_extractor.embedding_dim}')

## 🎬 **8. Extract Embeddings from One Video**

In [ ]:
def extract_video_embeddings(video_path, yolo_detector, feature_extractor,
                              max_frames=30, frame_skip=5):
    """
    Extract temporal embeddings from a single video.
    
    Strategy:
    - Sample every frame_skip-th frame
    - For each sampled frame: detect face → extract AffectNet embedding
    - If no face detected: use zero vector (keeps temporal alignment)
    - Pad or truncate to exactly max_frames
    
    Args:
        video_path: Path to .avi or .mp4 file
        yolo_detector: YOLOFaceDetector
        feature_extractor: AffectNetViTExtractor
        max_frames: Fixed sequence length (default: 30)
        frame_skip: Sample every N frames (default: 5)
    
    Returns:
        embeddings: numpy array (max_frames, 768)
    """
    cap = cv2.VideoCapture(str(video_path))
    
    if not cap.isOpened():
        print(f'⚠️  Cannot open: {video_path}')
        return np.zeros((max_frames, feature_extractor.embedding_dim), dtype=np.float32)
    
    embeddings = []
    frame_idx = 0
    no_face_count = 0
    
    while len(embeddings) < max_frames:
        ret, frame = cap.read()
        if not ret:
            break
        
        # Sample every frame_skip frames
        if frame_idx % frame_skip == 0:
            face_crop = yolo_detector.detect_and_crop(frame)
            
            if face_crop is not None:
                emb = feature_extractor.extract(face_crop)  # (768,)
            else:
                # No face detected — use zero vector to preserve temporal alignment
                emb = np.zeros(feature_extractor.embedding_dim, dtype=np.float32)
                no_face_count += 1
            
            embeddings.append(emb)
        
        frame_idx += 1
    
    cap.release()
    
    # Pad with zeros if video shorter than max_frames
    while len(embeddings) < max_frames:
        embeddings.append(np.zeros(feature_extractor.embedding_dim, dtype=np.float32))
    
    # Truncate if longer
    embeddings = embeddings[:max_frames]
    
    result = np.array(embeddings, dtype=np.float32)  # (max_frames, 768)
    
    return result


print('✅ extract_video_embeddings function defined')

## 🧪 **9. Test on One Video (Sanity Check)**

In [ ]:
# Test on the first video to make sure everything works
test_entry = all_videos[0]
print(f'Testing on: {test_entry["video_path"]}')
print(f'Labels: engagement={test_entry["engagement"]}, boredom={test_entry["boredom"]}, '
      f'confusion={test_entry["confusion"]}, frustration={test_entry["frustration"]}')

test_emb = extract_video_embeddings(
    test_entry['video_path'],
    yolo_detector,
    affectnet_extractor,
    max_frames=30,
    frame_skip=5
)

print(f'\nEmbedding shape: {test_emb.shape}')    # Should be (30, 768)
print(f'Dtype: {test_emb.dtype}')
print(f'Non-zero frames: {np.any(test_emb != 0, axis=1).sum()} / 30')
print(f'Mean value (non-zero frames): {test_emb[np.any(test_emb != 0, axis=1)].mean():.4f}')
print('\n✅ Sanity check passed! Shape is (30, 768) as expected.')

## 🚀 **10. Full Extraction Pipeline**

In [ ]:
def extract_all_embeddings(video_entries, yolo_detector, feature_extractor,
                            save_dir='/content/daisee_embeddings_affectnet',
                            max_frames=30, frame_skip=5):
    """
    Extract and save embeddings for all DAISEE videos.
    
    Saves:
        {save_dir}/{clip_name}.npy  — shape (30, 768)
        {save_dir}/metadata.json   — list of dicts with paths & labels
    
    Args:
        video_entries: List of dicts from find_videos_with_labels()
        yolo_detector: YOLOFaceDetector
        feature_extractor: AffectNetViTExtractor
        save_dir: Directory to save embeddings
        max_frames: Sequence length per video
        frame_skip: Frame sampling rate
    
    Returns:
        metadata: List of dicts (ready for Stage B EmbeddingDataset)
    """
    save_path = Path(save_dir)
    save_path.mkdir(parents=True, exist_ok=True)
    
    metadata = []
    errors = []
    
    print(f'Extracting embeddings for {len(video_entries)} videos...')
    print(f'Save directory: {save_dir}')
    print(f'Embedding shape per video: ({max_frames}, {feature_extractor.embedding_dim})')
    print('='*60)
    
    for idx, entry in enumerate(tqdm(video_entries, desc='Extracting')):
        clip_name = entry['clip_name']
        emb_path = save_path / f'{clip_name}.npy'
        
        # Skip if already extracted (resume support)
        if emb_path.exists():
            meta_entry = {
                'embedding_path': str(emb_path),
                'clip_name':      clip_name,
                'engagement':     entry['engagement'],
                'boredom':        entry['boredom'],
                'confusion':      entry['confusion'],
                'frustration':    entry['frustration'],
                'split':          entry['split']
            }
            metadata.append(meta_entry)
            continue
        
        try:
            emb = extract_video_embeddings(
                entry['video_path'],
                yolo_detector,
                feature_extractor,
                max_frames=max_frames,
                frame_skip=frame_skip
            )
            
            # Save embedding
            np.save(str(emb_path), emb)
            
            meta_entry = {
                'embedding_path': str(emb_path),
                'clip_name':      clip_name,
                'engagement':     entry['engagement'],
                'boredom':        entry['boredom'],
                'confusion':      entry['confusion'],
                'frustration':    entry['frustration'],
                'split':          entry['split']
            }
            metadata.append(meta_entry)
            
        except Exception as e:
            print(f'⚠️  Error on {clip_name}: {e}')
            errors.append(clip_name)
            continue
        
        # Save checkpoint every 100 videos
        if (idx + 1) % 100 == 0:
            checkpoint_path = save_path / 'metadata_checkpoint.json'
            with open(checkpoint_path, 'w') as f:
                json.dump(metadata, f)
            print(f'  💾 Checkpoint saved: {idx+1} videos processed')
    
    # Save final metadata
    metadata_path = save_path / 'metadata.json'
    with open(metadata_path, 'w') as f:
        json.dump(metadata, f, indent=2)
    
    print('\n' + '='*60)
    print(f'✅ Extraction complete!')
    print(f'   Total extracted: {len(metadata)}')
    print(f'   Errors: {len(errors)}')
    print(f'   Metadata saved: {metadata_path}')
    
    # Split summary
    splits = {}
    for item in metadata:
        s = item['split']
        splits[s] = splits.get(s, 0) + 1
    print(f'   Split distribution: {splits}')
    
    return metadata


print('✅ extract_all_embeddings function defined')

## ▶️ **11. RUN EXTRACTION — Execute This Cell**

⚠️ **This will take 2–5 hours depending on GPU speed.**  
Resume is supported — already-saved `.npy` files are skipped.

In [ ]:
SAVE_DIR = '/content/daisee_embeddings_affectnet'
MAX_FRAMES = 30    # Sequence length fed to LSTM
FRAME_SKIP = 5     # Sample every 5th frame

metadata = extract_all_embeddings(
    video_entries=all_videos,
    yolo_detector=yolo_detector,
    feature_extractor=affectnet_extractor,
    save_dir=SAVE_DIR,
    max_frames=MAX_FRAMES,
    frame_skip=FRAME_SKIP
)

print(f'\nMetadata ready: {len(metadata)} entries')
print(f'Embedding shape per entry: ({MAX_FRAMES}, {affectnet_extractor.embedding_dim})')

## ✅ **12. Verify & Show Statistics**

In [ ]:
import matplotlib.pyplot as plt

# Load metadata if not in memory
import json
with open(f'{SAVE_DIR}/metadata.json') as f:
    metadata = json.load(f)

print(f'Total clips: {len(metadata)}')

# Verify shapes
print('\nVerifying embedding shapes (checking 10 random samples)...')
import random
samples = random.sample(metadata, min(10, len(metadata)))
all_ok = True
for s in samples:
    emb = np.load(s['embedding_path'])
    if emb.shape != (MAX_FRAMES, affectnet_extractor.embedding_dim):
        print(f'  ❌ Wrong shape: {s["clip_name"]} -> {emb.shape}')
        all_ok = False
if all_ok:
    print(f'  ✅ All shapes correct: ({MAX_FRAMES}, {affectnet_extractor.embedding_dim})')

# Label distribution
df = pd.DataFrame(metadata)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
emotions = ['engagement', 'boredom', 'confusion', 'frustration']

for ax, emotion in zip(axes, emotions):
    counts = df[emotion].value_counts().sort_index()
    ax.bar(counts.index, counts.values, color='steelblue', edgecolor='black')
    ax.set_title(emotion.capitalize())
    ax.set_xlabel('Level (0–3)')
    ax.set_ylabel('Count')
    for i, v in zip(counts.index, counts.values):
        ax.text(i, v + 5, str(v), ha='center', fontsize=9)

plt.suptitle('DAISEE Label Distribution (All Splits)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/label_distribution.png', dpi=100)
plt.show()

print('\n✅ Stage A complete! Embeddings ready for Stage B training.')
print(f'📁 Files saved in: {SAVE_DIR}')
print(f'📄 Load in Stage B with: metadata.json')

## 📋 **13. Stage B Integration — How to Load These Embeddings**

In your Stage B notebook, replace the dataset loading with:

```python
import json
import numpy as np

EMBEDDING_DIR = '/content/daisee_embeddings_affectnet'

with open(f'{EMBEDDING_DIR}/metadata.json') as f:
    metadata = json.load(f)

# Split by 'split' field
train_meta = [m for m in metadata if m['split'] == 'train']
val_meta   = [m for m in metadata if m['split'] == 'val']
test_meta  = [m for m in metadata if m['split'] == 'test']

print(f'Train: {len(train_meta)} | Val: {len(val_meta)} | Test: {len(test_meta)}')
```

Then feed `train_meta`, `val_meta`, `test_meta` into your `EmbeddingDataset`.

**Embedding shape: `(30, 768)`** → Same as before, no changes needed to LSTM/KAN/CORAL.